# ML Literacy Cheat Sheet — Week 1

*Working version. Fill the markdown cells in your own words, then check against your Week 1 review doc to catch gaps. The empty code cells are optional demos — write a few lines to watch each concept run. Keep written entries to one line; if you condense this into a one-page printable later, use two columns / landscape.*

## 1. ML Basics

- Supervised learning: these are where models are trained on a labeled dataset
  - Classification: how a model can identify each data point in a categorical sense
  - Regression: how a  model can predict an outcome based on the data (continuous)
- Unsupervised learning: these models are trained with unlabled datasets and without predfined outcomes or human guidance
- Core workflow: raw data &rarr; preprocess data if needed &rarr; train/test split data &rarr; train model &rarr; test model with test data &rarr; create metrics based on model performance

## 2. Algorithms

| Algorithm | What it is | Strength | Weakness | When to use | Security use case |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Decision Tree** | A flow-chart-like model used for classification and regression. It splits the data on feature thresholds to reach a prediction. | Easy to interpret and visualize. | Prone to overfitting; high variance (small data changes can reshape the tree). | When relationships are non-linear and interpretability matters. | Triaging alerts where an analyst needs to see *why* something was flagged; building explainable rules for endpoint or email filtering. |
| **Random Forest** | An ensemble of many decision trees whose votes are averaged to classify or predict. | Reduces the overfitting and variance of a single tree; gives feature-importance scores. | More compute- and memory-intensive; less interpretable than one tree. | When you need a strong, low-effort baseline on high-dimensional or tabular data. | Malware classification, network-intrusion detection, and fraud detection on tabular feature sets. |
| **Logistic Regression** | A linear model that outputs a probability for binary (or multi-class) classification. | Fast, easy to implement, and the coefficients are interpretable. | Assumes a roughly linear decision boundary; often needs manual feature engineering. | When the target is categorical and a simple, explainable baseline is wanted. | Phishing/spam detection and a fast first-pass classifier before heavier models. |

In [1]:
# Demo for Section 2: train Decision Tree / Random Forest / Logistic Regression
# on the Iris dataset and print accuracy + a classification report for each.

#Imports
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Iris: 150 samples, 4 numeric features, 3 balanced classes (50 each).
X, y = load_iris(return_X_y=True)

# 80/20 split; random_state fixes the split so the same rows are held out every run.
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

# Random_state on the tree/forest makes their internal randomness reproducible.
# Max_iter on LogReg just gives the optimizer enough room to converge.
models = [
    DecisionTreeClassifier(random_state=42),
    RandomForestClassifier(random_state=42),
    LogisticRegression(max_iter=200),
]

for m in models:
    m.fit(Xtr, ytr)                       # Learn patterns from the training data
    name = type(m).__name__               # Prints models name
    preds = m.predict(Xte)                # Predict on the held-out test set
    print("=" * 60)
    print(f"{name}  |  accuracy = {m.score(Xte, yte):.3f}")
    print("=" * 60)
    # Precision / recall / f1 per class, plus overall accuracy + averages
    print(classification_report(yte, preds))

DecisionTreeClassifier  |  accuracy = 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

RandomForestClassifier  |  accuracy = 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

LogisticRegression  |  accuracy = 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
 

## 3. Metrics

**Confusion matrix** — each quadrant labeled with TP / FP / FN / TN and its security meaning:

|  | Predicted Positive (attack) | Predicted Negative (benign) |
| :--- | :--- | :--- |
| **Actual Positive (attack)** | **TP** — attack correctly detected | **FN** — missed attack (most costly) |
| **Actual Negative (benign)** | **FP** — false alarm on benign traffic | **TN** — benign correctly cleared |

| Metric | One-line intuition | Formula | Security note |
| :--- | :--- | :--- | :--- |
| **Precision** | Of everything flagged positive, how much was truly positive. | $TP / (TP + FP)$ | High precision = few false alarms; analysts trust the alerts and waste less time chasing benign traffic. |
| **Recall** | Of all real positives, how many were caught. | $TP / (TP + FN)$ | High recall = few missed attacks; this is what you protect when a miss is the costly outcome. |
| **F1** | Harmonic mean of precision and recall (balances the two). | $2 \cdot (Precision \cdot Recall) / (Precision + Recall)$ | A single score for detectors when both false alarms and misses matter. |
| **Accuracy** | Of all predictions, how many were correct. | $(TP + TN) / (TP + TN + FP + FN)$ | Misleading on imbalanced security data — a model that calls everything benign still scores high. |
| **Balanced Accuracy** | Average of recall on each class (handles class imbalance). |  | Fairer than accuracy when attacks are rare; rewards catching the minority (attack) class. |
| **ROC-AUC** | Probability the model ranks a random positive above a random negative, across all thresholds. |  | Threshold-independent view of separability; can look optimistic when benign vastly outnumbers attacks. |
| **PR-AUC** | Area under the precision-recall curve, summarizing the two across thresholds. |  | Preferred over ROC-AUC for rare attacks — it focuses on positive-class performance instead of abundant true negatives. |

In [2]:
# Demo for Section 3: a confusion matrix + classification report, then a look at
# WHY accuracy is misleading on imbalanced data (and what to read instead).
# Self-contained. Class 1 = "attack" (positive), class 0 = "benign" (negative).

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)

# ---------------------------------------------------------------------------
# Part A -- Confusion matrix + report on a BALANCED binary problem.
# sklearn orders labels ascending (0 then 1), so the matrix reads:
#     [[TN, FP],
#      [FN, TP]]      with positive class = 1 = "attack"
# ---------------------------------------------------------------------------
Xb, yb = make_classification(
    n_samples=2000, n_features=10, n_informative=5,
    weights=[0.5, 0.5], class_sep=0.8, random_state=42,
)
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
preds = clf.predict(Xte)

print("PART A -- balanced binary problem")
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(yte, preds))
print()
print(classification_report(yte, preds, target_names=["benign", "attack"]))

# ---------------------------------------------------------------------------
# Part B -- IMBALANCED problem: attacks are rare (~5%).
# Watch accuracy stay high while recall / PR-AUC tell the real story.
# ---------------------------------------------------------------------------
Xi, yi = make_classification(
    n_samples=5000, n_features=10, n_informative=4,
    weights=[0.95, 0.05], class_sep=0.6, random_state=42,
)
Xtr, Xte, ytr, yte = train_test_split(
    Xi, yi, test_size=0.3, random_state=42, stratify=yi
)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
preds = clf.predict(Xte)
proba = clf.predict_proba(Xte)[:, 1]   # P(attack), needed for ROC-AUC / PR-AUC

print("\nPART B -- imbalanced problem")
print(f"Attack prevalence in test set: {yte.mean():.1%}")
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(yte, preds))
print()
print(f"Accuracy          : {accuracy_score(yte, preds):.3f}   <- looks great...")
print(f"Balanced accuracy : {balanced_accuracy_score(yte, preds):.3f}")
print(f"Recall (attacks)  : {recall_score(yte, preds):.3f}   <- ...but attacks slip through")
print(f"ROC-AUC           : {roc_auc_score(yte, proba):.3f}")
print(f"PR-AUC            : {average_precision_score(yte, proba):.3f}   <- the honest score for rare positives")

# The cheat-sheet's "calls everything benign still scores high" case, made literal:
always_benign = np.zeros_like(yte)
print("\nBaseline that predicts 'benign' for EVERYTHING:")
print(f"Accuracy          : {accuracy_score(yte, always_benign):.3f}   <- high, yet useless")
print(f"Recall (attacks)  : {recall_score(yte, always_benign, zero_division=0):.3f}   <- catches zero attacks")
print("\nTakeaway: on rare-event data, judge detectors by recall / balanced accuracy / PR-AUC, not accuracy.")

PART A -- balanced binary problem
Confusion matrix [[TN, FP], [FN, TP]]:
[[197  86]
 [ 69 248]]

              precision    recall  f1-score   support

      benign       0.74      0.70      0.72       283
      attack       0.74      0.78      0.76       317

    accuracy                           0.74       600
   macro avg       0.74      0.74      0.74       600
weighted avg       0.74      0.74      0.74       600


PART B -- imbalanced problem
Attack prevalence in test set: 5.4%
Confusion matrix [[TN, FP], [FN, TP]]:
[[1419    0]
 [  81    0]]

Accuracy          : 0.946   <- looks great...
Balanced accuracy : 0.500
Recall (attacks)  : 0.000   <- ...but attacks slip through
ROC-AUC           : 0.679
PR-AUC            : 0.151   <- the honest score for rare positives

Baseline that predicts 'benign' for EVERYTHING:
Accuracy          : 0.946   <- high, yet useless
Recall (attacks)  : 0.000   <- catches zero attacks

Takeaway: on rare-event data, judge detectors by recall / balanced a

## 4. Principles / Gotchas

- Overfitting vs underfitting: 
    - Overfitting: When the model learns the training data too well instead of learning the underlying patterns in the dataset
    - Underfitting: When the model is too simple and can not learn the underlying patterns in the dataset
- Class imbalance -- effect on metrics + what to do: 
    - Class imbalance: When one category outnumbers the others which causes the model to favor that class and ignore all other category
    - Effects on metrics: skews the evaluation of the model where accuracy, precision, and ROC-AUC are affected by this
    - What to do: Use Recall, F1-Score, PR-AUC to determine how the model is performing
- Comparing models reliably: 
- When feature scaling matters (and when it doesn't): 
    - Matters: If the features in the dataset have a drastic range or unit difference
    - Does not Matter: Tree based models, strict interpretability, or if you know the min/max or mean, cluster algorithmns
- Reading a perfect / near-perfect score: 
    - Perfect: Proceed with caution due to the fact that there is a chance of some sort of data leakage is happening and would need to review the model as overfitting is likely occuring 
    - Near-perfect: Proceed with caution there is a chance that the model is just favoring the majority class and would need to review the confusion matrix or recall per class or PR-AUC to confirm model performance 